# ICL (Few-shot) Classification: Heatmap Builder

Heatmaps of ICL accuracy and logit diff as a function of
dropout rate (x) and noise SD (y).

## Experiment structure

Each ICL run has **1000 stochastic samples**: 500 DROPOUT
and 500 NOISE (2-class, balanced). Unlike zero-shot
classification A/B, ICL sweeps do NOT override
`active_perturbations`, so both perturbation types are
present in every run.

The sweep grid varies:
- `perturbation.dropout_rate` (11 values, per model range)
- `perturbation.noise_std` (11 values, per model range)
- `prompts.turns.num_pairs` (1, 3, 5, 7, 9 teaching examples)
- `prompts.turns.swap_labels` (true, false)

## Metrics and sample sizes

- **accuracy_primary**: correct / 1000 (all stochastic
  rows). SE = $\sqrt{p(1-p)/1000}$, SD = $\sqrt{p(1-p)}$.
- **logit_diff_correct**: logit(correct class) minus
  logit(incorrect class), averaged per group.
  For DROPOUT samples (n=500):
  `dropout_mean_logit_diff_dropout` = mean of
  (logit_dropout $-$ logit_noise).
  For NOISE samples (n=500):
  `noise_mean_logit_diff_noise` = mean of
  (logit_noise $-$ logit_dropout).
  SE and SD are empirical: `np.std / sqrt(500)` and
  `np.std`, both using population SD (divides by n).

## swap_labels

Every run scores its accuracy against the true perturbation, also when
`swap_labels=True` (the teaching pairs carry the wrong label). The heatmaps show:
- **swap=False**: accuracy as reported.
- **swap=True (flipped)**: accuracy w.r.t. the swapped (taught) labels, i.e. the
  fraction of *label* predictions that name the swapped label rather than the
  true perturbation. For the constrained rules every prediction is a label, so
  this is $100 - \text{accuracy}$; for the argmax rules $100 - \text{accuracy}$
  would also credit non-label (OTHER) predictions, so the label rate is used instead.
- **diff**: swap=False minus swap=True(flipped).

In [ ]:
import os
import pathlib

_this_dir = pathlib.Path(os.path.abspath("")).resolve()
if _this_dir.name == "paper":
    os.chdir(_this_dir.parent)

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from wandb_cache import load_sweep

In [ ]:
import yaml

# Sweeps: the ICL entries of the manifest sweeps.yaml (project llm-dropout-noise), prompt
# icl/main_zs: iclzs_main (k = 0..9, correct labels), iclzs_swap (swapped labels) and iclzs_ctrl (aliases at k = 1),
# each with its refills iclzs_<name>_<suffix>. The correct and swapped runs form one frame (df_main).
_manifest = yaml.safe_load(
    (_this_dir / "sweeps.yaml").read_text()
)  # cell 1 defines _this_dir
PROJECT = _manifest["project"]


def _icl_ids(name):
    base = f"iclzs_{name}"
    return [
        sid
        for ids in _manifest.values()
        if isinstance(ids, dict)
        for k, sid in ids.items()
        if k == base or k.startswith(base + "_")
    ]


MAIN_SWEEPS = _icl_ids("main") + _icl_ids("swap")
CONTROL_SWEEPS = _icl_ids("ctrl")

# --- Columns ---
COL_MODEL = "model"
COL_DROPOUT = "perturbation.dropout_rate"
COL_NOISE = "perturbation.noise_std"
COL_SWAP = "prompts.turns.swap_labels"
COL_NUM_PAIRS = "prompts.turns.num_pairs"
COL_ACC = "accuracy_constrained_primary"
COL_ALIASES = "aliases"

# --- Models ---
MODELS = [
    "llama3_8b",
    "qwen3_14b",
    "qwen3_32b",
    "olmo3_32b",
    "qwen3_8_27b",
    "seed_oss_36b",
    "mistral_small_24b",
    "gemma4_31b",
]

# LaTeX model names: \model{<key>} is defined in the paper's math_commands.tex
MODEL_MACROS_LATEX = {
    "llama3_8b": r"\model{llama8}",
    "qwen3_14b": r"\model{qwen14}",
    "qwen3_32b": r"\model{qwen32}",
    "olmo3_32b": r"\model{olmo32}",
    "qwen3_8_27b": r"\model{qwen27}",
    "seed_oss_36b": r"\model{seed36}",
    "mistral_small_24b": r"\model{mistral24}",
    "gemma4_31b": r"\model{gemma31}",
}

# --- Sample sizes (verified from sweep configs and code) ---
# ICL has 500 DROPOUT + 500 NOISE = 1000 stochastic per run.
N_STOCHASTIC = 1000
N_PER_GROUP = 500

In [ ]:
# --- Load data ---
def load_and_merge(sweep_ids, project):
    version = 2  # every run must carry metrics_version 2
    dfs = [
        load_sweep(sid, project=project, require_version=version) for sid in sweep_ids
    ]
    return pd.concat(dfs, ignore_index=True)


print("Main sweeps:")
df_main = load_and_merge(MAIN_SWEEPS, PROJECT)
df_main = df_main[df_main[COL_MODEL].isin(MODELS)].copy()
print(f"  {len(df_main)} runs")
print(f"  Models: {sorted(df_main[COL_MODEL].unique())}")
print(f"  num_pairs: {sorted(df_main[COL_NUM_PAIRS].dropna().unique())}")

print("\nControl sweeps:")
df_ctrl = load_and_merge(CONTROL_SWEEPS, PROJECT)
df_ctrl = df_ctrl[df_ctrl[COL_MODEL].isin(MODELS)].copy()
print(f"  {len(df_ctrl)} runs")
print(f"  Aliases: {sorted(df_ctrl[COL_ALIASES].dropna().unique())}")

# Select one rule available for every main and control run, including after
# concatenating sweeps of different metric versions. Never drop historical rows
# from a comparison just because their exact-token accuracy was not recorded.
LABEL_RATE_COL = {
    "accuracy_constrained_primary": None,
    "accuracy_primary": "argmax_is_primary_label_rate",
    "accuracy_aggregate": "argmax_is_label_rate",
}


def select_accuracy_metric(*frames):
    """Prefer exact-token accuracy; otherwise use the shared constrained rule."""
    if not frames:
        raise ValueError(
            "Main and control runs are required for an accuracy comparison"
        )
    for col in ("accuracy_primary", "accuracy_constrained_primary"):
        required = [col]
        if LABEL_RATE_COL[col] is not None:
            required.append(LABEL_RATE_COL[col])
        if all(
            all(key in df.columns and df[key].notna().all() for key in required)
            for df in frames
        ):
            return col
    raise ValueError("No accuracy rule is recorded for every main and control run")


COL_ACC = select_accuracy_metric(df_main, df_ctrl)
ACC_LABEL = COL_ACC  # menus show the W&B metric key, as in 0_localization / 1_zero_shot
print(f"Accuracy column for main/control comparisons: {COL_ACC}")
if COL_ACC != "accuracy_primary":
    print(
        "Exact-token accuracy or its label rate is missing for some loaded runs. "
        "Using constrained-primary accuracy for the entire comparison; "
        "this does not reconstruct historical exact-token accuracy."
    )

# --- Verify sample counts from logged data ---
for label, df in [("Main", df_main), ("Control", df_ctrl)]:
    if df.empty:  # control arm not run yet
        continue
    if "total_rows" in df.columns and "num_baselines" in df.columns:
        n_stoch = (df["total_rows"] - df["num_baselines"]).mode().iloc[0]
        print(
            f"\n{label}: total_rows - num_baselines = {int(n_stoch)} "
            f"(expected {N_STOCHASTIC})"
        )
    if "num_dropout" in df.columns:
        print(f"  num_dropout mode: {int(df['num_dropout'].mode().iloc[0])}")
    if "num_noise" in df.columns:
        print(f"  num_noise mode: {int(df['num_noise'].mode().iloc[0])}")

# --- Show per-model ranges ---
for name, col in [("Dropout", COL_DROPOUT), ("Noise", COL_NOISE)]:
    print(f"\n{name} ranges per model:")
    for model in MODELS:
        vals = sorted(df_main[df_main[COL_MODEL] == model][col].unique())
        if vals:
            print(f"  {model}: {vals[0]:.4f} .. {vals[-1]:.4f} ({len(vals)} points)")

In [ ]:
# --- Preprocess: add derived columns ---
# For each df, add columns that can be selected as heatmap z-values.
#
# COL_ACC and accuracy_aggregate are already present (fraction, 0-1).
# We compute SE and SD from Bernoulli formula with n=1000; for swap=True runs
# from the accuracy w.r.t. the swapped labels (see swapped_correct).
#
# For logit_diff: the evaluation logs per-group stats.
# logit_diff_correct = average of the "correct class" logit
# diff across both groups.
# For DROPOUT samples: logit_diff_dropout = logit(D) - logit(N)
# For NOISE samples:   logit_diff_noise   = logit(N) - logit(D)
# Both are "logit(correct) - logit(incorrect)".

# --- Check which logit diff columns exist ---
_LD_D = "dropout_mean_logit_diff_dropout"  # n=500
_LD_D_SE = "dropout_se_logit_diff_dropout"
_LD_D_SD = "dropout_std_logit_diff_dropout"
_LD_N = "noise_mean_logit_diff_noise"  # n=500
_LD_N_SE = "noise_se_logit_diff_noise"
_LD_N_SD = "noise_std_logit_diff_noise"

for label, df in [("Main", df_main), ("Control", df_ctrl)]:
    has_ld = _LD_D in df.columns and _LD_N in df.columns
    print(f"{label}: logit_diff columns present = {has_ld}")

    if has_ld:
        # logit_diff_correct: average of the two group means.
        # Both represent logit(correct) - logit(incorrect).
        df["logit_diff_correct"] = (df[_LD_D] + df[_LD_N]) / 2

        # SE of average of two independent means:
        # SE = sqrt(SE_d^2 + SE_n^2) / 2
        df["logit_diff_correct_se"] = np.sqrt(df[_LD_D_SE] ** 2 + df[_LD_N_SE] ** 2) / 2

        # SD: approximate as RMS of within-group SDs
        # (ignores between-group mean difference)
        df["logit_diff_correct_sd"] = np.sqrt(
            (df[_LD_D_SD] ** 2 + df[_LD_N_SD] ** 2) / 2
        )


# --- Bernoulli SE and SD for accuracy ---
def swapped_correct(df, col):
    """Accuracy w.r.t. the swapped (taught) labels for swap=True runs.

    ``df[col]`` is scored against the true perturbation, so the label predictions
    it counts as wrong are those that name the swapped label. ``1 - acc`` would
    also credit OTHER (non-label) predictions."""
    rate_col = LABEL_RATE_COL[col]
    label_rate = 1.0 if rate_col is None else df[rate_col]
    return label_rate - df[col]


for df in [df_main, df_ctrl]:
    swapped = df[COL_SWAP] == True
    for col, out in [
        (COL_ACC, "accuracy"),
        ("accuracy_aggregate", "accuracy_aggregate"),
    ]:
        p = df[col].where(~swapped, swapped_correct(df, col))
        pq = p * (1 - p)
        df[f"{out}_se"] = np.sqrt(pq / N_STOCHASTIC)
        df[f"{out}_sd"] = np.sqrt(pq)

# Default the views to swap=False when the manifest has no swapped-label runs.
HAS_SWAP = bool((df_main[COL_SWAP] == True).any())
print(f"swap=True runs present: {HAS_SWAP}")

# --- Available z-value metrics ---
Z_METRICS = {
    f"{ACC_LABEL} (%)": {
        "col": COL_ACC,
        "scale": 100,
        "flip_swap": True,  # swap=True uses the appropriate label rate
    },
    f"{ACC_LABEL} SE (%)": {
        "col": "accuracy_se",
        "scale": 100,
        "flip_swap": False,
    },
    f"{ACC_LABEL} SD (%)": {
        "col": "accuracy_sd",
        "scale": 100,
        "flip_swap": False,
    },
    "accuracy_aggregate (%)": {
        "col": "accuracy_aggregate",
        "scale": 100,
        "flip_swap": True,
    },
    "accuracy_aggregate SE (%)": {
        "col": "accuracy_aggregate_se",
        "scale": 100,
        "flip_swap": False,
    },
    "accuracy_aggregate SD (%)": {
        "col": "accuracy_aggregate_sd",
        "scale": 100,
        "flip_swap": False,
    },
}

if "logit_diff_correct" in df_main.columns:
    Z_METRICS["logit_diff_correct"] = {
        "col": "logit_diff_correct",
        "scale": 1,
        "flip_swap": True,  # swap flips the sign
    }
    Z_METRICS["logit_diff_correct SE"] = {
        "col": "logit_diff_correct_se",
        "scale": 1,
        "flip_swap": False,
    }
    Z_METRICS["logit_diff_correct SD"] = {
        "col": "logit_diff_correct_sd",
        "scale": 1,
        "flip_swap": False,
    }

print(f"\nAvailable z-metrics: {list(Z_METRICS.keys())}")

In [ ]:
# ── Heatmap helpers ──────────────────────────────────────────


def make_single_heatmap(df, z_col, scale, flip_swap, swap_value):
    """Build one panel, transforming each run before averaging across the grid."""
    selected = df[df[COL_SWAP] == swap_value].copy()
    if swap_value and flip_swap:
        selected[z_col] = (
            swapped_correct(selected, z_col) if scale == 100 else -selected[z_col]
        )
    return (
        selected.pivot_table(
            index=COL_NOISE,
            columns=COL_DROPOUT,
            values=z_col,
            aggfunc="mean",
        ).sort_index(ascending=False)
        * scale
    )


def make_heatmap_data(df, z_col, scale, flip_swap):
    """Use the same transformation for the three-panel and single-swap views."""
    val_f = make_single_heatmap(df, z_col, scale, flip_swap, False)
    val_t = make_single_heatmap(df, z_col, scale, flip_swap, True)
    return val_f, val_t, val_f - val_t


def plot_heatmaps(
    heat_f,
    heat_t,
    heat_diff,
    title_prefix,
    z_label,
    flip_swap,
    diff_vmin=np.nan,
    diff_vmax=np.nan,
):
    """Three-panel heatmap: swap=False, swap=True, diff."""
    n_cols = len(heat_f.columns)
    n_rows = len(heat_f.index)
    annot = max(n_cols, n_rows) <= 15
    annot_kws = {"size": 8} if annot else {}

    fig, axes = plt.subplots(1, 3, figsize=(22, max(5, n_rows * 0.5 + 2)))

    # Determine colormaps and ranges
    is_acc = "%" in z_label
    if is_acc:
        cmap_main = "RdYlGn"
        vmin_main, vmax_main = 0, 100
    else:
        cmap_main = "RdYlGn"
        vmin_main, vmax_main = None, None

    for ax, heat, swap_label in [
        (axes[0], heat_f, "swap=False"),
        (axes[1], heat_t, "swap=True (flipped)" if flip_swap else "swap=True"),
    ]:
        sns.heatmap(
            heat,
            annot=annot,
            fmt=".1f",
            cmap=cmap_main,
            vmin=vmin_main,
            vmax=vmax_main,
            ax=ax,
            cbar_kws={"label": z_label},
            annot_kws=annot_kws,
        )
        ax.set_title(f"{title_prefix}{swap_label}")
        ax.set_xlabel("dropout rate")
        ax.set_ylabel("noise std")

    # Diff panel
    auto_vmax = max(
        abs(heat_diff.min().min()),
        abs(heat_diff.max().max()),
        0.1,
    )
    vmin_d = -auto_vmax if np.isnan(diff_vmin) else diff_vmin
    vmax_d = auto_vmax if np.isnan(diff_vmax) else diff_vmax
    sns.heatmap(
        heat_diff,
        annot=annot,
        fmt=".1f",
        cmap="RdBu_r",
        center=0,
        vmin=vmin_d,
        vmax=vmax_d,
        ax=axes[2],
        cbar_kws={"label": "diff"},
        annot_kws=annot_kws,
    )
    axes[2].set_title(f"{title_prefix}diff (F \u2212 T)")
    axes[2].set_xlabel("dropout rate")
    axes[2].set_ylabel("noise std")

    plt.tight_layout()
    plt.show()


# ── LaTeX generation ─────────────────────────────────────────


def generate_single_heatmap_latex(
    heat,
    cmap_name,
    vmin,
    vmax,
    title="",
    show_ylabel=True,
    show_colorbar=True,
    show_xlabel=True,
    rotate_xticks=True,
    title_yshift=None,
    cell_font=r"\tiny",
):
    """Generate pgfplots LaTeX for one heatmap panel.

    show_xlabel / show_ylabel / show_colorbar / rotate_xticks / title_yshift let a caller
    reproduce the paper's multi-panel figures, where labels and colourbars depend on the
    panel's position in the grid (paper Fig. 6: xlabel bottom row only, ylabel left column
    only, colourbar right column only, unrotated x ticks, title shifted down).
    """
    rows_sorted = sorted(heat.index.tolist())
    cols = heat.columns.tolist()
    n_rows, n_cols = len(rows_sorted), len(cols)
    row_idx = {v: i for i, v in enumerate(rows_sorted)}
    col_idx = {v: i for i, v in enumerate(cols)}

    def _end_labels(n, lo, hi):
        return ",".join(lo if i == 0 else hi if i == n - 1 else "" for i in range(n))

    lines = []
    lines.append(r"\begin{tikzpicture}")
    lines.append(r"\begin{axis}[")
    lines.append("    width=0.82\\linewidth,")  # paper Fig. 5 geometry: square panel
    lines.append("    height=0.82\\linewidth,")
    lines.append(
        "    xlabel={\\small Dropout rate $p$}," if show_xlabel else "    xlabel={},"
    )
    if show_ylabel:
        lines.append("    ylabel={\\small Noise SD $\\sigma$},")
    else:
        lines.append(r"    ylabel={},")
        lines.append(r"    yticklabels={},")
    if show_colorbar:
        lines.append("    colorbar,")
    if cmap_name == "pastel_rg":
        lines.append(
            "    colormap={pastel_rwg}{rgb255(0cm)=(206,143,143); rgb255(50cm)=(255,255,255); rgb255(100cm)=(143,206,143)},"
        )
    elif (
        cmap_name == "pastel_po"
    ):  # delta heatmaps: pastel orange (negative), white, pastel purple (positive)
        lines.append(
            "    colormap={pastel_po}{rgb255(0cm)=(225,160,100); rgb255(50cm)=(255,255,255); rgb255(100cm)=(160,135,200)},"
        )
    elif cmap_name == "RdBu_r":
        lines.append(
            "    colormap={rdbu}{rgb255(0cm)=(33,102,172); rgb255(50cm)=(255,255,255); rgb255(100cm)=(178,24,43)},"
        )
    else:
        lines.append(f"    colormap name={cmap_name},")
    lines.append(f"    point meta min={vmin:.1f},")
    lines.append(f"    point meta max={vmax:.1f},")
    xticks = ",".join(str(i) for i in range(n_cols))
    xticklabels = _end_labels(n_cols, "$p_{\\min}$", "$p_{\\max}$")
    yticks = ",".join(str(i) for i in range(n_rows))
    yticklabels_str = _end_labels(n_rows, "$\\sigma_{\\min}$", "$\\sigma_{\\max}$")
    lines.append(f"    xtick={{{xticks}}},")
    lines.append(f"    xticklabels={{{xticklabels}}},")
    lines.append(f"    ytick={{{yticks}}},")
    if show_ylabel:
        lines.append(f"    yticklabels={{{yticklabels_str}}},")
    lines.append(
        r"    x tick label style={font=\scriptsize, rotate=45},"
        if rotate_xticks
        else r"    x tick label style={font=\scriptsize},"
    )
    lines.append(r"    y tick label style={font=\scriptsize},")
    lines.append(r"    view={0}{90},")
    lines.append(r"    axis line style={draw=none},")
    lines.append(r"    major tick length=0pt,")
    lines.append(r"    enlargelimits=false,")
    if show_colorbar:
        lines.append(r"    colorbar style={font=\tiny},")
    if title:
        lines.append(r"    title style={font=\small},")
        lines.append(f"    title={{{title}}},")
        if title_yshift:
            lines.append(f"    title style={{yshift={title_yshift}}},")
    lines.append(r"]")

    rows_desc = list(reversed(rows_sorted))
    lines.append(
        r"\addplot[matrix plot*, mesh/cols="
        + str(n_cols)
        + r", mesh/rows="
        + str(n_rows)
        + r", point meta=explicit] table[meta=C] {"
    )
    lines.append("x y C")
    for row_val in rows_desc:
        for col_val in cols:
            val = heat.loc[row_val, col_val]
            lines.append(f"{col_idx[col_val]} {row_idx[row_val]} {val:.2f}")
    lines.append("};")

    for row_val in rows_sorted:
        for col_val in cols:
            val = heat.loc[row_val, col_val]
            lines.append(
                f"\\node[font={cell_font}] at (axis cs:{col_idx[col_val]},{row_idx[row_val]}) {{{val:.1f}}};"
            )

    lines.append(r"\end{axis}")
    lines.append(r"\end{tikzpicture}")
    return "\n".join(lines)


def generate_three_heatmaps_latex(heat_f, heat_t, heat_diff, label, flip_swap):
    """LaTeX figure with 3 heatmaps: swap=F, swap=T, diff."""
    panel_f = generate_single_heatmap_latex(
        heat_f,
        "pastel_rg",
        0,
        100,
        title="swap=False",
        show_ylabel=True,
        show_colorbar=False,
    )
    panel_t = generate_single_heatmap_latex(
        heat_t,
        "pastel_rg",
        0,
        100,
        title="swap=True" + (" (flipped)" if flip_swap else ""),
        show_ylabel=False,
        show_colorbar=True,
    )
    auto_v = max(abs(heat_diff.min().min()), abs(heat_diff.max().max()), 1)
    panel_d = generate_single_heatmap_latex(
        heat_diff,
        "RdBu_r",
        -auto_v,
        auto_v,
        title="diff",
        show_ylabel=False,
        show_colorbar=True,
    )

    safe_label = label.replace("_", chr(92) + "_")
    caption = f"ICL heatmaps for {safe_label}: swap=False (left), swap=True (center), difference (right)."
    return (
        "\\begin{figure}[ht]\n"
        "\\centering\n"
        f"\\begin{{minipage}}{{0.33\\textwidth}}%\n"
        f"  {panel_f}\n"
        f"\\end{{minipage}}%\n"
        f"\\begin{{minipage}}{{0.33\\textwidth}}%\n"
        f"  {panel_t}\n"
        f"\\end{{minipage}}%\n"
        f"\\begin{{minipage}}{{0.33\\textwidth}}%\n"
        f"  {panel_d}\n"
        f"\\end{{minipage}}\n"
        f"\\caption{{{caption}}}\n"
        f"\\label{{fig:icl_heatmap_{label}}}\n"
        "\\end{figure}"
    )

## Main Experiments

In [ ]:
# ── Interactive heatmap: Main ────────────────────────────────

main_models = sorted(df_main[COL_MODEL].unique())
main_num_pairs = sorted(df_main[COL_NUM_PAIRS].dropna().unique())

w_model = widgets.Dropdown(options=main_models, description="Model:")
w_np = widgets.SelectionSlider(
    options=[(str(int(v)), v) for v in main_num_pairs],
    description="num_pairs:",
    layout=widgets.Layout(width="300px"),
)
w_z = widgets.Dropdown(
    options=list(Z_METRICS.keys()),
    value=list(Z_METRICS)[0],
    description="z metric:",
)
w_swap_mode = widgets.RadioButtons(
    options=["Three-panel (F / T / diff)", "swap=False only", "swap=True only"],
    value="Three-panel (F / T / diff)" if HAS_SWAP else "swap=False only",
    description="Swap:",
    layout=widgets.Layout(width="auto"),
)
w_dvmin = widgets.FloatText(
    value=float("nan"),
    description="diff vmin:",
    layout=widgets.Layout(width="160px"),
)
w_dvmax = widgets.FloatText(
    value=float("nan"),
    description="diff vmax:",
    layout=widgets.Layout(width="160px"),
)
out_main = widgets.Output()


def redraw_main(*_):
    out_main.clear_output(wait=True)
    with out_main:
        model = w_model.value
        np_val = w_np.value
        z_key = w_z.value
        z_cfg = Z_METRICS[z_key]
        swap_mode = w_swap_mode.value

        subset = df_main[
            (df_main[COL_MODEL] == model) & (df_main[COL_NUM_PAIRS] == np_val)
        ]
        print(f"Model={model}, num_pairs={int(np_val)}: {len(subset)} runs, z={z_key}")

        if subset.empty or z_cfg["col"] not in subset.columns:
            print("No data for this selection.")
            return

        if swap_mode == "Three-panel (F / T / diff)":
            hf, ht, hd = make_heatmap_data(
                subset,
                z_cfg["col"],
                z_cfg["scale"],
                z_cfg["flip_swap"],
            )
            plot_heatmaps(
                hf,
                ht,
                hd,
                title_prefix=f"{model} | {int(np_val)} pairs | ",
                z_label=z_key,
                flip_swap=z_cfg["flip_swap"],
                diff_vmin=w_dvmin.value,
                diff_vmax=w_dvmax.value,
            )
            # LaTeX
            if "%" in z_key:
                print(
                    generate_three_heatmaps_latex(
                        hf,
                        ht,
                        hd,
                        f"main_{model}_np{int(np_val)}",
                        z_cfg["flip_swap"],
                    )
                )
        else:
            # Single-swap heatmap
            swap_val = swap_mode == "swap=True only"
            heat = make_single_heatmap(
                subset, z_cfg["col"], z_cfg["scale"], z_cfg["flip_swap"], swap_val
            )

            n_cols, n_rows = len(heat.columns), len(heat.index)
            annot = max(n_cols, n_rows) <= 15
            is_acc = "%" in z_key

            fig, ax = plt.subplots(
                figsize=(max(8, n_cols * 0.7 + 3), max(5, n_rows * 0.5 + 2))
            )
            sns.heatmap(
                heat,
                annot=annot,
                fmt=".1f",
                cmap="RdYlGn",
                vmin=0 if is_acc else None,
                vmax=100 if is_acc else None,
                ax=ax,
                cbar_kws={"label": z_key},
                annot_kws={"size": 8} if annot else {},
            )
            swap_lbl = (
                "swap=True (flipped)"
                if swap_val and z_cfg["flip_swap"]
                else f"swap={swap_val}"
            )
            ax.set_title(f"{model} | {int(np_val)} pairs | {swap_lbl}")
            ax.set_xlabel("dropout rate")
            ax.set_ylabel("noise std")
            plt.tight_layout()
            plt.show()

            # LaTeX (single panel)
            if "%" in z_key:
                _title = MODEL_MACROS_LATEX[model] + (
                    "" if swap_val is False else " (swapped labels)"
                )
                _panel = generate_single_heatmap_latex(
                    heat, "pastel_rg", 0, 100, title=_title
                )
                _safe = f"main_{model}_np{int(np_val)}".replace("_", chr(92) + "_")
                print(
                    "\\begin{figure}[ht]\n"
                    "\\centering\n"
                    f"{_panel}\n"
                    f"\\caption{{ICL heatmap for {_safe} ({swap_lbl}).}}\n"
                    f"\\label{{fig:icl_single_{model}_np{int(np_val)}}}\n"
                    "\\end{figure}"
                )


for w in [w_model, w_np, w_z, w_swap_mode, w_dvmin, w_dvmax]:
    w.observe(redraw_main, names="value")

display(
    widgets.HBox([w_model, w_np, w_z]),
    widgets.HBox([w_swap_mode, w_dvmin, w_dvmax]),
    out_main,
)
redraw_main()

## Control Experiments

Same grid but with various alias labels. Select a model,
num_pairs, alias, and z-metric.

In [ ]:
# ── Interactive heatmap: Controls ────────────────────────────

# Include main experiments as "dropout_noise" alias
_df_main_tagged = df_main.copy()
_df_main_tagged[COL_ALIASES] = "dropout_noise"
df_ctrl_all = pd.concat([df_ctrl, _df_main_tagged], ignore_index=True)

ctrl_models = sorted(df_ctrl_all[COL_MODEL].unique())
ctrl_num_pairs = sorted(df_ctrl_all[COL_NUM_PAIRS].dropna().unique())
ctrl_aliases = sorted(df_ctrl_all[COL_ALIASES].dropna().unique())

cw_model = widgets.Dropdown(options=ctrl_models, description="Model:")
cw_np = widgets.SelectionSlider(
    options=[(str(int(v)), v) for v in ctrl_num_pairs],
    description="num_pairs:",
    layout=widgets.Layout(width="300px"),
)
cw_alias = widgets.Dropdown(options=ctrl_aliases, description="Alias:")
cw_z = widgets.Dropdown(
    options=list(Z_METRICS.keys()),
    value=list(Z_METRICS)[0],
    description="z metric:",
)
cw_swap_mode = widgets.RadioButtons(
    options=[
        "Three-panel (F / T / diff)",
        "swap=False only",
        "swap=True only",
        "diff only",
    ],
    value="Three-panel (F / T / diff)" if HAS_SWAP else "swap=False only",
    description="Swap:",
    layout=widgets.Layout(width="auto"),
)
cw_dvmin = widgets.FloatText(
    value=float("nan"),
    description="diff vmin:",
    layout=widgets.Layout(width="160px"),
)
cw_dvmax = widgets.FloatText(
    value=float("nan"),
    description="diff vmax:",
    layout=widgets.Layout(width="160px"),
)
out_ctrl = widgets.Output()


def redraw_ctrl(*_):
    out_ctrl.clear_output(wait=True)
    with out_ctrl:
        model = cw_model.value
        np_val = cw_np.value
        alias = cw_alias.value
        z_key = cw_z.value
        z_cfg = Z_METRICS[z_key]
        swap_mode = cw_swap_mode.value

        subset = df_ctrl_all[
            (df_ctrl_all[COL_MODEL] == model)
            & (df_ctrl_all[COL_NUM_PAIRS] == np_val)
            & (df_ctrl_all[COL_ALIASES] == alias)
        ]
        print(
            f"Model={model}, num_pairs={int(np_val)}, "
            f"alias={alias}: {len(subset)} runs, z={z_key}"
        )

        if subset.empty or z_cfg["col"] not in subset.columns:
            print("No data for this selection.")
            return

        prefix = f"{model} | {int(np_val)} pairs | {alias} | "

        if swap_mode == "Three-panel (F / T / diff)":
            hf, ht, hd = make_heatmap_data(
                subset,
                z_cfg["col"],
                z_cfg["scale"],
                z_cfg["flip_swap"],
            )
            plot_heatmaps(
                hf,
                ht,
                hd,
                title_prefix=prefix,
                z_label=z_key,
                flip_swap=z_cfg["flip_swap"],
                diff_vmin=cw_dvmin.value,
                diff_vmax=cw_dvmax.value,
            )
            # LaTeX
            if "%" in z_key:
                print(
                    generate_three_heatmaps_latex(
                        hf,
                        ht,
                        hd,
                        f"ctrl_{alias}_{model}_np{int(np_val)}",
                        z_cfg["flip_swap"],
                    )
                )
        elif swap_mode == "diff only":
            hf, ht, hd = make_heatmap_data(
                subset,
                z_cfg["col"],
                z_cfg["scale"],
                z_cfg["flip_swap"],
            )
            n_cols = len(hd.columns)
            n_rows = len(hd.index)
            annot = max(n_cols, n_rows) <= 15
            auto_vmax = max(
                abs(hd.min().min()),
                abs(hd.max().max()),
                0.1,
            )
            vmin_d = -auto_vmax if np.isnan(cw_dvmin.value) else cw_dvmin.value
            vmax_d = auto_vmax if np.isnan(cw_dvmax.value) else cw_dvmax.value

            fig, ax = plt.subplots(
                figsize=(max(8, n_cols * 0.7 + 3), max(5, n_rows * 0.5 + 2))
            )
            sns.heatmap(
                hd,
                annot=annot,
                fmt=".1f",
                cmap="RdBu_r",
                center=0,
                vmin=vmin_d,
                vmax=vmax_d,
                ax=ax,
                cbar_kws={"label": "diff"},
                annot_kws={"size": 8} if annot else {},
            )
            ax.set_title(f"{prefix}diff (F \u2212 T)")
            ax.set_xlabel("dropout rate")
            ax.set_ylabel("noise std")
            plt.tight_layout()
            plt.show()

            # LaTeX (diff panel only)
            if "%" in z_key:
                _panel = generate_single_heatmap_latex(
                    hd,
                    "RdBu_r",
                    vmin_d,
                    vmax_d,
                    title=r"diff (F $-$ T)",
                )
                _safe = f"ctrl_{alias}_{model}_np{int(np_val)}".replace(
                    "_", chr(92) + "_"
                )
                print(
                    "\\begin{figure}[ht]\n"
                    "\\centering\n"
                    f"{_panel}\n"
                    f"\\caption{{ICL control diff heatmap for {_safe}.}}\n"
                    f"\\label{{fig:icl_ctrl_diff_{model}_{alias}}}\n"
                    "\\end{figure}"
                )
        else:
            swap_val = swap_mode == "swap=True only"
            heat = make_single_heatmap(
                subset, z_cfg["col"], z_cfg["scale"], z_cfg["flip_swap"], swap_val
            )

            n_cols, n_rows = len(heat.columns), len(heat.index)
            annot = max(n_cols, n_rows) <= 15
            is_acc = "%" in z_key

            fig, ax = plt.subplots(
                figsize=(max(8, n_cols * 0.7 + 3), max(5, n_rows * 0.5 + 2))
            )
            sns.heatmap(
                heat,
                annot=annot,
                fmt=".1f",
                cmap="RdYlGn",
                vmin=0 if is_acc else None,
                vmax=100 if is_acc else None,
                ax=ax,
                cbar_kws={"label": z_key},
                annot_kws={"size": 8} if annot else {},
            )
            swap_lbl = (
                "swap=True (flipped)"
                if swap_val and z_cfg["flip_swap"]
                else f"swap={swap_val}"
            )
            ax.set_title(f"{prefix}{swap_lbl}")
            ax.set_xlabel("dropout rate")
            ax.set_ylabel("noise std")
            plt.tight_layout()
            plt.show()

            # LaTeX (single panel)
            if "%" in z_key:
                _title = swap_lbl.replace("_", r"\_")
                _panel = generate_single_heatmap_latex(
                    heat, "pastel_rg", 0, 100, title=_title
                )
                _safe = f"ctrl_{alias}_{model}_np{int(np_val)}".replace(
                    "_", chr(92) + "_"
                )
                print(
                    "\\begin{figure}[ht]\n"
                    "\\centering\n"
                    f"{_panel}\n"
                    f"\\caption{{ICL control heatmap for {_safe} ({swap_lbl}).}}\n"
                    f"\\label{{fig:icl_ctrl_single_{model}_{alias}}}\n"
                    "\\end{figure}"
                )


for w in [cw_model, cw_np, cw_alias, cw_z, cw_swap_mode, cw_dvmin, cw_dvmax]:
    w.observe(redraw_ctrl, names="value")

display(
    widgets.HBox([cw_model, cw_np, cw_alias, cw_z]),
    widgets.HBox([cw_swap_mode, cw_dvmin, cw_dvmax]),
    out_ctrl,
)
redraw_ctrl()

## Paper Fig. 6: delta heatmaps, one panel per alias

For one model at k=1: accuracy with correct labels minus accuracy with swapped labels (the notebook's
flipped difference, as in the paper), for the four aliases of the paper figure. Prints one panel per
alias in the paper's file format (`plots/02_few_shots/02_delta_<alias>.tex`): xlabel only on the bottom
row, ylabel only on the left column, colourbar only on the right column, fixed $\pm$25 scale.


In [ ]:
# ── Paper Fig. 6 emitter ─────────────────────────────────────
FIG6_ALIASES = [
    "dropout_noise",
    "masking_jitter",
    "x_y",
    "rotation_permutation",
]  # paper order (row-major 2x2)
FIG6_TITLES = {
    "dropout_noise": "Dropout/Noise",
    "masking_jitter": "Masking/Jitter",
    "x_y": "X/Y",
    "rotation_permutation": "Rotation/Permutation",
}
FIG6_FILES = {
    "dropout_noise": "02_delta_main",
    "masking_jitter": "02_delta_masking_jitter",
    "x_y": "02_delta_x_y",
    "rotation_permutation": "02_delta_rotation_permutation",
}
FIG6_VLIM = 25.0
FIG6_CELL_FONT = r"\fontsize{4}{4.8}\selectfont"  # one step below \tiny (5pt) for the 11x11 cell numbers

f6_w_model = widgets.Dropdown(
    options=ctrl_models,
    value="qwen3_32b" if "qwen3_32b" in ctrl_models else ctrl_models[0],
    description="Model:",
)
f6_w_np = widgets.Dropdown(
    options=[(str(int(v)), v) for v in ctrl_num_pairs],
    value=ctrl_num_pairs[0],
    description="num_pairs:",
)
f6_w_alias = widgets.Dropdown(
    options=[("all four panels", "all")] + [(FIG6_TITLES[a], a) for a in FIG6_ALIASES],
    value="dropout_noise",
    description="Panel:",
)
f6_out = widgets.Output()


def emit_fig6(*_):
    f6_out.clear_output(wait=True)
    with f6_out:
        z = Z_METRICS[f"{COL_ACC} (%)"]
        for i, alias in enumerate(FIG6_ALIASES):
            if f6_w_alias.value != "all" and alias != f6_w_alias.value:
                continue
            sub = df_ctrl_all[
                (df_ctrl_all[COL_MODEL] == f6_w_model.value)
                & (df_ctrl_all[COL_NUM_PAIRS] == f6_w_np.value)
                & (df_ctrl_all[COL_ALIASES] == alias)
            ]
            if sub.empty:
                print(
                    f"% {alias}: no runs for {f6_w_model.value} at k={int(f6_w_np.value)}"
                )
                continue
            _, _, hd = make_heatmap_data(sub, z["col"], z["scale"], z["flip_swap"])
            row, col = divmod(i, 2)
            panel = generate_single_heatmap_latex(
                hd,
                "RdBu_r",
                -FIG6_VLIM,
                FIG6_VLIM,
                title=FIG6_TITLES[alias],
                show_xlabel=(row == 1),
                show_ylabel=(col == 0),
                show_colorbar=(col == 1),
                rotate_xticks=False,
                title_yshift="-1.0ex",
                cell_font=FIG6_CELL_FONT,
            )
            print(
                f"% ===== plots/02_few_shots/{FIG6_FILES[alias]}.tex  ({f6_w_model.value}, k={int(f6_w_np.value)}) ====="
            )
            print("\\centering\n" + panel + "\n")


for w in (f6_w_model, f6_w_np, f6_w_alias):
    w.observe(emit_fig6, names="value")
display(widgets.HBox([f6_w_model, f6_w_np, f6_w_alias]), f6_out)
emit_fig6()

## Learning Dynamics

Mean accuracy (and SE/SD) vs number of teaching examples,
one curve per model. For each (model, num_pairs), we take
the 121 heatmap cells (11 dropout × 11 noise) with
swap=False. Each cell is one run with $n = 1000$ samples.

### SE and SD computation

We treat the 121 runs as 121 × 1000 = 121,000 independent
Bernoulli trials. The grand mean accuracy is:

$$\bar{p} = \frac{1}{121} \sum_{i=1}^{121} p_i$$

which equals the pooled proportion (all runs have the same
$n$). The Bernoulli SE and SD of this pooled estimate are:

$$\text{SE} = \sqrt{\frac{\bar{p}(1-\bar{p})}{121000}}
\qquad
\text{SD} = \sqrt{\bar{p}(1-\bar{p})}$$

**Caveat**: this assumes all 121,000 trials are i.i.d.
In reality the true accuracy varies across the 121
(dropout, noise) configurations. The Bernoulli SE
therefore **understates** the uncertainty. For comparison,
the empirical SE across the 121 run means is
$\text{std}(p_1, \ldots, p_{121}) / \sqrt{121}$, which
captures both sampling noise and between-configuration
variability. Both are shown in the debug output.

In [ ]:
# ── Learning dynamics: compute table ─────────────────────────

MODEL_COLORS = {  # same palette as 0_localization
    "llama3_8b": "#f4b6c2",
    "qwen3_14b": "#8fae82",
    "qwen3_32b": "#1b3a6b",
    "olmo3_32b": "#8b1a1a",
    "qwen3_8_27b": "#00a3a3",
    "seed_oss_36b": "#b5651d",
    "mistral_small_24b": "#d4a017",
    "gemma4_31b": "#4b86b4",
}
MODEL_COLORS_LATEX = {
    "llama3_8b": "red!40",
    "qwen3_14b": "green!40!black",
    "qwen3_32b": "blue!70!black",
    "olmo3_32b": "red!70!black",
    "qwen3_8_27b": "teal!70!black",
    "seed_oss_36b": "brown!80!black",
    "mistral_small_24b": "yellow!60!black",
    "gemma4_31b": "cyan!60!black",
}

N_GRID = 121  # 11 dropout x 11 noise


def compute_learning_table(df, swap_value):
    """Compute mean accuracy and logit diff with errors
    for each (model, num_pairs) at a given swap_labels value.

    For accuracy (Bernoulli):
      - n_total = n_runs * 1000
      - pooled_se = sqrt(p*(1-p) / n_total)
      - sd = sqrt(p*(1-p))
    For logit_diff_correct (continuous):
      - pooled_se = mean(within_run_sd) / sqrt(n_total)
      - sd = mean(within_run_sd)
    Empirical SE (for both): std across run means / sqrt(n_runs).
    """
    sub = df[df[COL_SWAP] == swap_value]
    has_ld = "logit_diff_correct" in sub.columns
    rows = []
    for model in MODELS:
        ms = sub[sub[COL_MODEL] == model]
        for np_val in sorted(ms[COL_NUM_PAIRS].dropna().unique()):
            chunk = ms[ms[COL_NUM_PAIRS] == np_val]
            # swap=True rows: accuracy w.r.t. the swapped labels (see swapped_correct)
            acc = swapped_correct(chunk, COL_ACC) if swap_value else chunk[COL_ACC]
            n_runs = len(acc)
            if n_runs == 0:
                continue

            p_mean = acc.mean()
            pq = p_mean * (1 - p_mean)
            n_total = n_runs * N_STOCHASTIC

            # Argmax accuracy
            acc_argmax = (
                swapped_correct(chunk, "accuracy_aggregate")
                if swap_value
                else chunk["accuracy_aggregate"]
            )
            p_argmax = acc_argmax.mean()
            pq_argmax = p_argmax * (1 - p_argmax)

            row = {
                "model": model,
                "num_pairs": int(np_val),
                "swap": swap_value,
                "n_runs": n_runs,
                "n_total": n_total,
                "acc_pct": p_mean * 100,
                "acc_pooled_se_pct": np.sqrt(pq / n_total) * 100,
                "acc_sd_pct": np.sqrt(pq) * 100,
                "acc_empirical_se_pct": (acc.std() / np.sqrt(n_runs) * 100),
                "acc_argmax_pct": p_argmax * 100,
                "acc_argmax_pooled_se_pct": np.sqrt(pq_argmax / n_total) * 100,
                "acc_argmax_sd_pct": np.sqrt(pq_argmax) * 100,
                "acc_argmax_empirical_se_pct": (
                    acc_argmax.std() / np.sqrt(n_runs) * 100
                ),
            }

            if has_ld:
                ld = chunk["logit_diff_correct"]
                ld_sd_col = "logit_diff_correct_sd"
                ld_mean = ld.mean()
                within_sd = (
                    chunk[ld_sd_col].mean() if ld_sd_col in chunk.columns else np.nan
                )
                row.update(
                    {
                        "ld_mean": ld_mean,
                        "ld_pooled_se": within_sd / np.sqrt(n_total),
                        "ld_sd": within_sd,
                        "ld_empirical_se": (ld.std() / np.sqrt(n_runs)),
                    }
                )

            rows.append(row)
    return pd.DataFrame(rows)


# Compute for both swap values
ldt_false = compute_learning_table(df_main, swap_value=False)
ldt_true = compute_learning_table(df_main, swap_value=True)

# For swap=True the table was already computed w.r.t. the swapped labels
# (swapped_correct), so the "flipped" columns are the values themselves and
# their SE/SD were computed from the same flipped p.
if HAS_SWAP:
    ldt_true["acc_pct_flipped"] = ldt_true["acc_pct"]
    ldt_true["acc_argmax_pct_flipped"] = ldt_true["acc_argmax_pct"]
    ldt_true["acc_pooled_se_pct_flipped"] = ldt_true["acc_pooled_se_pct"]
    ldt_true["acc_empirical_se_pct_flipped"] = ldt_true["acc_empirical_se_pct"]
    ldt_true["acc_sd_pct_flipped"] = ldt_true["acc_sd_pct"]
    ldt_true["acc_argmax_pooled_se_pct_flipped"] = ldt_true["acc_argmax_pooled_se_pct"]
    ldt_true["acc_argmax_empirical_se_pct_flipped"] = ldt_true[
        "acc_argmax_empirical_se_pct"
    ]
    ldt_true["acc_argmax_sd_pct_flipped"] = ldt_true["acc_argmax_sd_pct"]
# For logit diff, negate (swap reverses correct/incorrect)
if "ld_mean" in ldt_true.columns:
    ldt_true["ld_mean_flipped"] = -ldt_true["ld_mean"]
    ldt_true["ld_pooled_se_flipped"] = ldt_true["ld_pooled_se"]
    ldt_true["ld_empirical_se_flipped"] = ldt_true["ld_empirical_se"]
    ldt_true["ld_sd_flipped"] = ldt_true["ld_sd"]

print("swap=False:", len(ldt_false), "rows")
print("swap=True:", len(ldt_true), "rows")
display(ldt_false.head(5))

In [ ]:
# ── Learning dynamics: interactive plot ──────────────────────
# swap=False curves are solid+thick (main).
# swap=True curves are dashed+thin (secondary).

ld_w_models = {
    m: widgets.Checkbox(value=True, description=m, indent=False) for m in MODELS
}
ld_w_metric = widgets.RadioButtons(
    options=[
        (ACC_LABEL, "selected_accuracy"),
        ("accuracy_aggregate", "aggregate_accuracy"),
        ("logit_diff_correct", "logit_diff"),
    ],
    value="selected_accuracy",
    description="Metric:",
    layout=widgets.Layout(width="auto"),
)
ld_w_error = widgets.RadioButtons(
    options=["Pooled SE", "Empirical SE", "SD"],
    value="Pooled SE",
    description="Band:",
    layout=widgets.Layout(width="auto"),
)
ld_w_swap = widgets.RadioButtons(
    options=["swap=False only", "swap=True only", "Both"],
    value="Both" if HAS_SWAP else "swap=False only",
    description="Swap:",
    layout=widgets.Layout(width="auto"),
)
ld_w_ymin = widgets.FloatText(
    value=float("nan"),
    description="y min:",
    layout=widgets.Layout(width="150px"),
)
ld_w_ymax = widgets.FloatText(
    value=float("nan"),
    description="y max:",
    layout=widgets.Layout(width="150px"),
)
ld_out = widgets.Output()


def _ld_cols(met, et, flipped=False):
    """Return (y_col, err_col) for the given metric/error/flip."""
    sfx = "_flipped" if flipped else ""
    if met == "aggregate_accuracy":
        y_col = f"acc_argmax_pct{sfx}"
        if et == "Pooled SE":
            err_col = f"acc_argmax_pooled_se_pct{sfx}"
        elif et == "Empirical SE":
            err_col = f"acc_argmax_empirical_se_pct{sfx}"
        else:
            err_col = f"acc_argmax_sd_pct{sfx}"
    elif met == "selected_accuracy":
        y_col = f"acc_pct{sfx}"
        if et == "Pooled SE":
            err_col = f"acc_pooled_se_pct{sfx}"
        elif et == "Empirical SE":
            err_col = f"acc_empirical_se_pct{sfx}"
        else:
            err_col = f"acc_sd_pct{sfx}"
    else:
        y_col = f"ld_mean{sfx}"
        if et == "Pooled SE":
            err_col = f"ld_pooled_se{sfx}"
        elif et == "Empirical SE":
            err_col = f"ld_empirical_se{sfx}"
        else:
            err_col = f"ld_sd{sfx}"
    return y_col, err_col


def redraw_ld(*_):
    ld_out.clear_output(wait=True)
    with ld_out:
        models = [m for m in MODELS if ld_w_models[m].value]
        met = ld_w_metric.value
        et = ld_w_error.value
        swap_mode = ld_w_swap.value
        ymin = ld_w_ymin.value if not np.isnan(ld_w_ymin.value) else None
        ymax = ld_w_ymax.value if not np.isnan(ld_w_ymax.value) else None

        is_pct = met in ("selected_accuracy", "aggregate_accuracy")
        ylabel = "Accuracy (%)" if is_pct else "Logit diff (correct \u2212 incorrect)"

        show_false = swap_mode in ("swap=False only", "Both")
        show_true = swap_mode in ("swap=True only", "Both")

        fig, ax = plt.subplots(figsize=(10, 6))

        from matplotlib.lines import Line2D

        legend_handles = []

        # --- swap=False: solid, thick ---
        if show_false:
            y_col, err_col = _ld_cols(met, et, flipped=False)
            if y_col not in ldt_false.columns:
                print(f"Column {y_col} not found in swap=False table.")
            else:
                for model in models:
                    sub = ldt_false[ldt_false["model"] == model].sort_values(
                        "num_pairs"
                    )
                    if sub.empty:
                        continue
                    x = sub["num_pairs"].values
                    y = sub[y_col].values
                    err = sub[err_col].values
                    c = MODEL_COLORS[model]
                    ax.plot(
                        x,
                        y,
                        marker="o",
                        markersize=5,
                        linewidth=2.5,
                        color=c,
                        zorder=10,
                    )
                    ax.fill_between(x, y - err, y + err, alpha=0.3, color=c, zorder=9)

        # --- swap=True (flipped): dashed, thin ---
        if show_true:
            y_col, err_col = _ld_cols(met, et, flipped=True)
            if y_col not in ldt_true.columns:
                print(f"Column {y_col} not found in swap=True table.")
            else:
                for model in models:
                    sub = ldt_true[ldt_true["model"] == model].sort_values("num_pairs")
                    if sub.empty:
                        continue
                    x = sub["num_pairs"].values
                    y = sub[y_col].values
                    err = sub[err_col].values
                    c = MODEL_COLORS[model]
                    ax.plot(
                        x,
                        y,
                        marker="o",
                        markersize=3,
                        linewidth=1.5,
                        color=c,
                        ls="--",
                        alpha=0.7,
                        zorder=5,
                    )
                    ax.fill_between(x, y - err, y + err, alpha=0.15, color=c, zorder=4)

        # Legend: model colors + line style key
        for model in models:
            legend_handles.append(
                Line2D([0], [0], color=MODEL_COLORS[model], lw=2, label=model)
            )
        if show_false and show_true:
            legend_handles.append(
                Line2D([0], [0], color="gray", lw=2.5, ls="-", label="swap=False")
            )
            legend_handles.append(
                Line2D(
                    [0], [0], color="gray", lw=1.5, ls="--", label="swap=True (flipped)"
                )
            )

        if is_pct:
            ax.axhline(50, color="gray", ls=":", alpha=0.5)
        else:
            ax.axhline(0, color="gray", ls=":", alpha=0.5)
        ax.set_xlabel("Number of teaching examples")
        ax.set_ylabel(ylabel)
        ax.set_xticks(sorted(ldt_false["num_pairs"].unique()))
        ax.grid(True, alpha=0.3)
        ax.legend(handles=legend_handles, fontsize=9)
        if ymin is not None and ymax is not None and ymin < ymax:
            ax.set_ylim(ymin, ymax)

        fig.suptitle(
            f"ICL Learning Dynamics: {ld_w_metric.label} (\u00b1{et})",
            fontsize=14,
        )
        fig.tight_layout()
        plt.show()

        # LaTeX
        _latex_lines = []
        _latex_lines.append(r"\begin{tikzpicture}")
        _latex_lines.append(r"\begin{axis}[")
        _latex_lines.append("    height=7cm,")
        _latex_lines.append("    width=\\linewidth,")
        _latex_lines.append("    grid=major,")
        _latex_lines.append("    xlabel={\\small Num of teaching pairs},")
        _latex_lines.append(
            f"    ylabel={{\\small {ylabel.replace(chr(37), chr(92) + chr(37))}}},"
        )  # escape % for LaTeX
        _all_np = sorted(ldt_false["num_pairs"].unique())
        _ticks = ",".join(str(int(v)) for v in _all_np)
        _latex_lines.append(f"    xtick={{{_ticks}}},")
        _latex_lines.append(r"    legend entries={},")
        _latex_lines.append(r"]")

        if show_false:
            _y_col, _err_col = _ld_cols(met, et, flipped=False)
            if _y_col in ldt_false.columns:
                for _m in models:
                    _sub = ldt_false[ldt_false["model"] == _m].sort_values("num_pairs")
                    if _sub.empty:
                        continue
                    _color = MODEL_COLORS_LATEX.get(_m, "gray")
                    _pts = list(zip(_sub["num_pairs"], _sub[_y_col], _sub[_err_col]))
                    _upper = " ".join(f"({int(x)},{y + e:.2f})" for x, y, e in _pts)
                    _lower = " ".join(
                        f"({int(x)},{y - e:.2f})" for x, y, e in reversed(_pts)
                    )
                    _latex_lines.append(
                        f"\\addplot[{_color}, fill={_color}, fill opacity=0.3, draw=none, forget plot] coordinates {{{_upper} {_lower}}} --cycle;"
                    )
                    _coords = " ".join(f"({int(x)},{y:.2f})" for x, y, _ in _pts)
                    _latex_lines.append(
                        f"\\addplot[{_color}, mark=*, mark size=1.5, line width=1.5pt, forget plot] coordinates {{{_coords}}};"
                    )

        if show_true:
            _y_col, _err_col = _ld_cols(met, et, flipped=True)
            if _y_col in ldt_true.columns:
                for _m in models:
                    _sub = ldt_true[ldt_true["model"] == _m].sort_values("num_pairs")
                    if _sub.empty:
                        continue
                    _color = MODEL_COLORS_LATEX.get(_m, "gray")
                    _pts = list(zip(_sub["num_pairs"], _sub[_y_col], _sub[_err_col]))
                    _upper = " ".join(f"({int(x)},{y + e:.2f})" for x, y, e in _pts)
                    _lower = " ".join(
                        f"({int(x)},{y - e:.2f})" for x, y, e in reversed(_pts)
                    )
                    _latex_lines.append(
                        f"\\addplot[{_color}, fill={_color}, fill opacity=0.15, draw=none, forget plot] coordinates {{{_upper} {_lower}}} --cycle;"
                    )
                    _coords = " ".join(f"({int(x)},{y:.2f})" for x, y, _ in _pts)
                    _latex_lines.append(
                        f"\\addplot[{_color}, dashed, mark=o, mark size=1, line width=1pt, forget plot] coordinates {{{_coords}}};"
                    )

        if is_pct:
            _latex_lines.append(
                f"\\addplot[gray, dashed, line width=0.5pt, forget plot] coordinates {{({int(_all_np[0])},50) ({int(_all_np[-1])},50)}};"
            )
        else:
            _latex_lines.append(
                f"\\addplot[gray, dashed, line width=0.5pt, forget plot] coordinates {{({int(_all_np[0])},0) ({int(_all_np[-1])},0)}};"
            )

        _latex_lines.append(r"\end{axis}")
        _latex_lines.append(r"\end{tikzpicture}")

        # paper legend: \scriptsize block, two models per row, filled markers
        _legend_items = []
        for _m in sorted(models):  # alphabetical legend
            _color = MODEL_COLORS_LATEX.get(_m, "gray")
            _label = "{\\small " + MODEL_MACROS_LATEX[_m] + "}"
            _legend_items.append(
                f"\\tikz\\draw[{_color}, thick, mark=*, mark size=1.5] plot coordinates {{(0,0) (0.4,0)}}; {_label}"
            )
        # centred rows of at most three models (the figure sits in a narrow minipage): each row is its
        # own tabular so the names' widths do not matter; a \\centering context centres the rows.
        _per_row = 3
        _legend_rows = [
            "\\begin{tabular}{@{}"
            + "l" * len(row)
            + "@{}}"
            + " & ".join(row)
            + "\\end{tabular}"
            for row in (
                _legend_items[i : i + _per_row]
                for i in range(0, len(_legend_items), _per_row)
            )
        ]
        _legend_line = "\\\\[2pt]\n".join(_legend_rows)

        print(
            "\\begin{figure}[ht]\n"
            "\\centering\n"
            f"{_legend_line}\\\\[4pt]\n" + "\n".join(_latex_lines) + "\n"
            f"\\caption{{ICL learning dynamics: {met.replace(chr(95), chr(92) + chr(95))}.}}\n"  # escape _ for LaTeX
            "\\label{fig:icl_learning_dynamics}\n"
            "\\end{figure}"
        )


for w in [ld_w_metric, ld_w_error, ld_w_swap, ld_w_ymin, ld_w_ymax]:
    w.observe(redraw_ld, names="value")
for cb in ld_w_models.values():
    cb.observe(redraw_ld, names="value")

display(
    widgets.HBox([ld_w_metric, ld_w_error, ld_w_swap, ld_w_ymin, ld_w_ymax]),
    widgets.HBox(list(ld_w_models.values())),
    ld_out,
)
redraw_ld()